In [2]:
# Loads the raw heart disease dataset, handles missing values,
# normalizes numerical features, encodes categorical variables,
# selects relevant features, and saves the cleaned dataset
import pandas as pd
from sklearn.preprocessing import MinMaxScaler,OneHotEncoder

In [3]:
df= pd.read_csv('C:/Users/ASUS/Downloads/habibas/IpProject2/data/heart.csv')
df.info()
df.sample(6)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1024 non-null   float64
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1023 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(2), int64(12)
memory usage: 112.2 KB


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
888,60,0,0,150,258,0,0.0,157,0,2.6,1,2,3,0
975,39,1,0,118,219,0,1.0,140,0,1.2,1,0,3,0
287,71,0,1,160,302,0,1.0,162,0,0.4,2,2,2,1
440,62,0,0,150,244,0,1.0,154,1,1.4,1,0,2,0
853,67,1,0,120,229,0,0.0,129,1,2.6,1,2,3,0
543,59,1,3,134,204,0,1.0,162,0,0.8,2,2,2,0


In [4]:
# Check for Missing Values
df.isnull().sum()

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     1
thalach     0
exang       0
oldpeak     2
slope       0
ca          0
thal        0
target      0
dtype: int64

In [5]:
# ── Check for Duplicate Rows
num_duplicates = df.duplicated().sum()
print(f'Number of duplicate rows: {num_duplicates}')

# Drop duplicate rows and keep the first occurrence
df.drop_duplicates(inplace=True)

# Confirm duplicates are removed
print(f'Duplicates after removal: {df.duplicated().sum()}')
print(f'Dataset shape after removing duplicates: {df.shape}')

Number of duplicate rows: 720
Duplicates after removal: 0
Dataset shape after removing duplicates: (305, 14)


In [6]:
# ── Handle Missing Values
# Fill missing restecg values with the most frequent value (mode)
df.fillna({'restecg': df['restecg'].mode()[0]}, inplace=True)
# Fill missing oldpeak values with the column mean
df.fillna({'oldpeak':df['oldpeak'].mean()},inplace=True)
# Confirm no missing values remain
df.isnull().sum()

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

In [ ]:
# Normalize Numerical Features
# Scale selected features to the range [0, 1] using Min-Max normalization
scaler=MinMaxScaler()
df[['age', 'trestbps', 'chol', 'thalach','oldpeak']]=scaler.fit_transform(df[['age', 'trestbps', 'chol', 'thalach','oldpeak']])


Scaler saved!


In [8]:
# Encode Categorical Variables
# Convert categorical columns into binary (one-hot) encoded columns using OneHotEncoder
cols = ['sex', 'cp', 'fbs', 'exang', 'restecg', 'slope', 'ca', 'thal']
encoder = OneHotEncoder(sparse_output=False, drop='first')
# Fit and transform the categorical columns
encoded_array = encoder.fit_transform(df[cols])
# Create a DataFrame with the new encoded column names
encoded_df = pd.DataFrame(
    encoded_array,
    columns=encoder.get_feature_names_out(cols),
    index=df.index
)

# Drop original categorical columns and replace with encoded ones
df = df.drop(columns=cols)
df = pd.concat([df, encoded_df], axis=1)

In [12]:
# Feature Selection — Correlation Analysis
# Compute correlation of each feature with the target variable
corr=df.corr()['target']
print(corr)

age           -0.222436
trestbps      -0.143093
thalach        0.419045
oldpeak       -0.430408
target         1.000000
sex_1         -0.284274
cp_1           0.247935
cp_2           0.314025
exang_1       -0.435867
restecg_1.0    0.174088
slope_1       -0.360740
slope_2        0.392221
ca_1          -0.234955
ca_2          -0.277773
ca_3          -0.207848
thal_2         0.510987
thal_3        -0.477569
Name: target, dtype: float64


In [13]:
# Drop features with very low correlation to the target (threshold: 0.1)
cols = corr[abs(corr) < 0.1].index.tolist()

df = df.drop(columns=cols)


In [14]:
df.sample(10)

,age,trestbps,thalach,oldpeak,target,sex_1,cp_1,cp_2,exang_1,restecg_1.0,slope_1,slope_2,ca_1,ca_2,ca_3,thal_2,thal_3
623,0.666667,0.377358,0.564885,0.419355,0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
287,0.875000,0.622642,0.694656,0.064516,1,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
328,0.854167,0.339623,0.290076,0.387097,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
144,0.375000,0.169811,0.549618,0.016129,1,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
146,0.458333,0.245283,0.656489,0.096774,1,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
719,0.479167,0.132075,0.580153,0.016129,1,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
429,0.375000,0.132075,0.618321,0.000000,0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
342,0.750000,0.575472,0.587786,0.129032,1,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
142,0.666667,0.377358,0.564885,0.419355,0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
377,0.729167,0.339623,0.389313,0.322581,1,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0


In [15]:
# Save the cleaned dataset for use in model training and the expert system
df.to_csv('C:/Users/ASUS/Downloads/habibas/IpProject2/data/cleaneddata.csv', index=False)